In [13]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

In [14]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.documents import Document
import tiktoken

In [15]:
def count_tokens(text, model="gpt-4o"):

    encoding = tiktoken.encoding_for_model(model)

    return len(encoding.encode(text))

In [16]:
documents = [
    Document(page_content="Subscription pricing works well for enterprise SaaS because it ensures predictable revenue and aligns with long-term contracts."),
    Document(page_content="Freemium models often fail in enterprise due to security, compliance, and onboarding complexity."),
    Document(page_content="Tiered pricing helps segment customers and increase revenue by aligning features with willingness to pay."),
    Document(page_content="Enterprise customers prefer reliability, support, and predictable billing over low-cost options."),
]

In [17]:
def build_context(docs):

    return "\n\n".join([doc.page_content for doc in docs])

In [18]:
def compress_context(context, max_tokens=200):

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=os.getenv("OPENAI_API_KEY"))

    messages = [
        SystemMessage(content="Summarize the following context while preserving key insights."),
        HumanMessage(content=context)
    ]

    response = llm.invoke(messages)

    return response.content

In [19]:
def manage_context_window(docs, max_tokens=200):

    context = build_context(docs)

    tokens = count_tokens(context)

    print(f"Original tokens: {tokens}")

    # If within limit → return as-is
    if tokens <= max_tokens:
        print("Within limit, no compression needed")
        return context

    print("Exceeds limit → compressing...")

    # Compress
    compressed = compress_context(context, max_tokens)

    print(f"Compressed tokens: {count_tokens(compressed)}")

    return compressed

In [20]:
def generate_answer(query, docs):

    context = manage_context_window(docs, max_tokens=150)

    llm = ChatOpenAI(model="gpt-4o", temperature=0, openai_api_key=os.getenv("OPENAI_API_KEY"))

    messages = [
        SystemMessage(content="You are a product strategist."),
        HumanMessage(content=f"""
Context:
{context}

Question:
{query}
""")
    ]

    return llm.invoke(messages).content

In [21]:
query = "What pricing strategy should we use for enterprise customers?"

response = generate_answer(query, documents)

print(response)

Original tokens: 70
Within limit, no compression needed
For enterprise customers, a subscription-based pricing strategy is generally the most effective approach. This strategy aligns well with the needs and preferences of enterprise clients for several reasons:

1. **Predictable Revenue**: Subscription pricing ensures a steady and predictable revenue stream, which is crucial for financial planning and stability. Enterprises often have long-term budgets and prefer predictable expenses.

2. **Long-term Contracts**: Enterprises typically engage in long-term contracts, and subscription models naturally align with this preference. This can lead to higher customer retention and reduced churn.

3. **Reliability and Support**: Enterprise customers value reliability and comprehensive support. A subscription model can include premium support services, ensuring that customers receive the assistance they need, which can be a significant differentiator.

4. **Tiered Pricing**: Implementing a tiered